# Error Analysis

Load a trained model from MLflow and analyze misclassified samples on both
train and validation sets to understand failure modes.

In [1]:
%matplotlib inline

import sys
sys.path.insert(0, '..')

import json
import torch
import mlflow
import mlflow.pytorch
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from omegaconf import OmegaConf

from src.dataset import TiledMultiChannelDataset, LabelEncoder
from src.transforms import Normalize, Compose, build_transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

c:\Users\kayhan\Anaconda3\envs\DL-project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## Configuration
Set only the run name and tracking URI. Everything else is loaded from artifacts.

In [2]:
# ========== CONFIGURE THESE ==========
RUN_NAME = "Vits_finetune_cosine_warmup_autoGradual_moredata_moresamples_augmentation"  # Name of the MLflow run
TRACKING_URI = "file:../mlruns"
EXPERIMENT_NAME = "image_classifier"  # MLflow experiment name
BATCH_SIZE = 64
# ======================================

In [3]:
# List all runs in the experiment
mlflow.set_tracking_uri(TRACKING_URI)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise ValueError(f"Experiment '{EXPERIMENT_NAME}' not found")

all_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
)

print(f"Found {len(all_runs)} runs in '{EXPERIMENT_NAME}':\n")
for i, row in all_runs.iterrows():
    name = row.get("tags.mlflow.runName", "unnamed")
    status = row.get("status", "?")
    start = str(row.get("start_time", ""))[:19]
    val_acc = row.get("metrics.val/acc", None)
    val_loss = row.get("metrics.val/loss", None)
    acc_str = f"val_acc={val_acc:.3f}" if val_acc is not None else "val_acc=N/A"
    loss_str = f"val_loss={val_loss:.3f}" if val_loss is not None else "val_loss=N/A"
    print(f"  [{i}] {name:60s} {acc_str}  {loss_str}  ({status}, {start})")

c:\Users\kayhan\Anaconda3\envs\DL-project\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Found 26 runs in 'image_classifier':

  [0] Vitb_finetune_cosine_warmup_autoGradual_moredata_moresamples_augmentation_moreLRdecay val_acc=0.584  val_loss=0.886  (FINISHED, 2026-05-12 16:04:17)
  [1] test                                                         val_acc=0.470  val_loss=1.136  (FINISHED, 2026-05-07 09:08:01)
  [2] Vitb_finetune_cosine_warmup_autoGradual_moredata_moresamples_augmentation val_acc=0.639  val_loss=1.043  (FINISHED, 2026-05-06 15:41:28)
  [3] Vits_finetune_cosine_warmup_autoGradual_moredata_moresamples_augmentation_LessLRdecay val_acc=0.661  val_loss=1.011  (FINISHED, 2026-05-06 13:46:32)
  [4] effb3_finetune_cosine_warmup_autoGradual_moredata_moresamples_augmentation_LessLRdecay_rerun val_acc=0.600  val_loss=1.067  (FINISHED, 2026-05-05 10:17:34)
  [5] Vits_finetune_cosine_warmup_autoGradual_moredata_moresamples_augmentation val_acc=0.632  val_loss=0.892  (FINISHED, 2026-05-04 17:57:07)
  [6] effb3_finetune_cosine_warmup_autoGradual_moredata_moresamples_augmen

## Load Model and Artifacts

In [4]:
# Select run and load artifacts
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"run_name = '{RUN_NAME}'",
    order_by=["start_time DESC"],
    max_results=1,
)
if runs.empty:
    raise ValueError(f"No run found with name '{RUN_NAME}' in experiment '{EXPERIMENT_NAME}'")

run_id = runs.iloc[0].run_id
print(f"Selected run: {RUN_NAME} (id={run_id[:8]}...)")

# Download artifacts
artifact_dir = mlflow.artifacts.download_artifacts(run_id=run_id, tracking_uri=TRACKING_URI)
print(f"Artifacts: {artifact_dir}")

# Load hydra config
config_path = Path(artifact_dir) / "hydra_config.yaml"
cfg = OmegaConf.load(config_path)
print(f"Model: {cfg.model.get('backbone_name', cfg.model.get('_target_'))}")

# Load dataset manifest
manifest_path = Path(artifact_dir) / "dataset_manifest.json"
with open(manifest_path, 'r') as f:
    manifest = json.load(f)

print(f"Train samples in manifest: {len(manifest['train_samples'])}")
print(f"Val samples in manifest: {len(manifest['val_samples'])}")

Selected run: Vits_finetune_cosine_warmup_autoGradual_moredata_moresamples_augmentation (id=d3c83be3...)
Artifacts: D:\personal_project\image_classifier\mlruns\320812426146031575\d3c83be374fa4eb1b713df9522eb154b\artifacts
Model: vit_small
Train samples in manifest: 64
Val samples in manifest: 36


In [5]:
# Load model from the run
# 1. Try runs:/{run_id}/model (new runs with name="model")
# 2. Try models:/ registry (old runs with artifact_path)
# 3. Fall back to checkpoint

model = None

# 1. Try run artifact
try:
    model_uri = f"runs:/{run_id}/model"
    print(f"Trying: {model_uri}")
    model = mlflow.pytorch.load_model(model_uri, map_location=device)
    print("Loaded from run artifact")
except Exception:
    pass

# 2. Try model registry (find version linked to this run)
if model is None:
    try:
        client = mlflow.MlflowClient()
        for mv in client.search_model_versions():
            if mv.run_id == run_id:
                model_uri = f"models:/{mv.name}/{mv.version}"
                print(f"Trying registry: {model_uri}")
                model = mlflow.pytorch.load_model(model_uri, map_location=device)
                print("Loaded from model registry")
                break
    except Exception as e:
        print(f"Registry lookup failed: {e}")

# 3. Fall back to checkpoint
if model is None:
    client = mlflow.MlflowClient()
    ckpt_artifacts = client.list_artifacts(run_id, path="checkpoints")
    if not ckpt_artifacts:
        raise FileNotFoundError(f"No model or checkpoint found in run {run_id}")
    ckpt_local = mlflow.artifacts.download_artifacts(
        run_id=run_id, artifact_path=ckpt_artifacts[0].path, tracking_uri=TRACKING_URI,
    )
    print(f"Loading from checkpoint: {ckpt_local}")
    model_target = cfg.model._target_
    if "TransferLearning" in model_target:
        from src.model import TransferLearningClassifier as ModelClass
    else:
        from src.model import CNNClassifier as ModelClass
    model = ModelClass.load_from_checkpoint(ckpt_local, map_location=device, weights_only=False)

model = model.to(device)
model.eval()
print(f"Model: {model.__class__.__name__}, Num classes: {model.num_classes}")

Trying: runs:/d3c83be374fa4eb1b713df9522eb154b/model


c:\Users\kayhan\Anaconda3\envs\DL-project\Lib\site-packages\mlflow\tracking\_model_registry\utils.py:220: FutureWarning: The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri)


Trying registry: models:/TransferLearningClassifier/20
Loaded from model registry
Model: TransferLearningClassifier, Num classes: 4


## Load Train and Validation Datasets
- **Train (no aug)**: Same train set with only Normalize â€” to see true model performance on train data
- **Train (with aug)**: Same train set with the exact training augmentations from config â€” to see the effect of augmentation on errors
- **Val (no aug)**: Validation set with only Normalize

In [ ]:
# Extract dataset config from hydra config
ds_cfg = cfg.datamodule.dataset
ROOT_DIR = ds_cfg.root_dir
CHANNELS = list(ds_cfg.channels)
CROP_SIZE = ds_cfg.crop_size
CACHE_SIZE = ds_cfg.get('cache_size', 64)

print(f"Root dir: {ROOT_DIR}")
print(f"Channels: {CHANNELS}")
print(f"Crop size: {CROP_SIZE}")

# Build label encoder from manifest labels
all_labels = sorted(set(
    s['label'] for s in manifest['train_samples'] + manifest['val_samples']
))
label_encoder = LabelEncoder()
label_encoder.fit(all_labels)
print(f"Classes: {label_encoder.classes}")
print(f"Num classes: {label_encoder.num_classes}")

# Build labels_dict from manifest (plate, well) -> label
def manifest_to_labels_dict(samples):
    """Convert manifest sample list to {(plate, well): label} dict."""
    labels = {}
    for s in samples:
        labels[(s['plate'], s['well'])] = s['label']
    return labels

train_labels_dict = manifest_to_labels_dict(manifest['train_samples'])
val_labels_dict = manifest_to_labels_dict(manifest['val_samples'])

print(f"\nTrain wells: {len(train_labels_dict)}")
print(f"Val wells: {len(val_labels_dict)}")

# Build transforms
no_aug_transform = Compose([Normalize()])

# Build train augmentation transform from saved hydra config (same as training)
train_transform_cfg = OmegaConf.to_container(cfg.datamodule.train_transform, resolve=True)
train_aug_transform = build_transforms(train_transform_cfg)
print(f"\nTrain augmentation pipeline: {train_aug_transform}")

# --- Train dataset WITHOUT augmentation ---
train_dataset = TiledMultiChannelDataset(
    root_dir=ROOT_DIR,
    channels=CHANNELS,
    crop_size=CROP_SIZE,
    stride=None,
    cache_size=CACHE_SIZE,
    max_samples_per_label=ds_cfg.get('max_samples_per_label', None),
    labels_dict=train_labels_dict,
    label_encoder=label_encoder,
    transform=no_aug_transform,
    verbose=True,
)

# --- Train dataset WITH augmentation (same transforms as training) ---
train_dataset_aug = TiledMultiChannelDataset(
    root_dir=ROOT_DIR,
    channels=CHANNELS,
    crop_size=CROP_SIZE,
    stride=None,
    cache_size=CACHE_SIZE,
    max_samples_per_label=ds_cfg.get('max_samples_per_label', None),
    labels_dict=train_labels_dict,
    label_encoder=label_encoder,
    transform=train_aug_transform,
    verbose=True,
)

# --- Val dataset (no augmentation) ---
val_dataset = TiledMultiChannelDataset(
    root_dir=ROOT_DIR,
    channels=CHANNELS,
    crop_size=CROP_SIZE,
    stride=None,
    cache_size=CACHE_SIZE,
    max_samples_per_label=ds_cfg.get('max_samples_per_label', None),
    labels_dict=val_labels_dict,
    label_encoder=label_encoder,
    transform=no_aug_transform,
    verbose=True,
)

print(f"\nTrain samples (no aug): {len(train_dataset)}")
print(f"Train samples (with aug): {len(train_dataset_aug)}")
print(f"Val samples (no aug): {len(val_dataset)}")

## Run Inference on Both Train Sets

In [ ]:
@torch.no_grad()
def evaluate_with_metadata(model, dataset, batch_size=64, device='cuda'):
    """Evaluate model and return per-tile predictions, labels, probs, and metadata.
    
    Returns:
        List of dicts, one per tile:
            - idx: tile index in the dataset
            - true_label: int label
            - pred_label: int label
            - confidence: float, probability of predicted class
            - true_conf: float, probability of true class
            - probs: np.array of all class probabilities
            - plate, well, field: sample metadata
            - top, left: tile position
    """
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    results = []
    tile_idx = 0
    
    for images, labels in loader:
        images = images.to(device)
        logits = model(images)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        labels_np = labels.numpy()
        
        for i in range(len(labels_np)):
            sample_idx, top, left = dataset.tiles[tile_idx]
            sample = dataset.samples[sample_idx]
            
            results.append({
                'idx': tile_idx,
                'true_label': int(labels_np[i]),
                'pred_label': int(preds[i]),
                'confidence': float(probs[i, preds[i]]),
                'true_conf': float(probs[i, labels_np[i]]),
                'probs': probs[i],
                'plate': sample['plate'],
                'well': sample['well'],
                'field': sample['field'],
                'top': top,
                'left': left,
            })
            tile_idx += 1
    
    return results

print("Evaluating Train (no aug)...")
train_results = evaluate_with_metadata(model, train_dataset, BATCH_SIZE, device)
train_acc = sum(r['true_label'] == r['pred_label'] for r in train_results) / len(train_results)

print("Evaluating Train (with aug)...")
train_aug_results = evaluate_with_metadata(model, train_dataset_aug, BATCH_SIZE, device)
train_aug_acc = sum(r['true_label'] == r['pred_label'] for r in train_aug_results) / len(train_aug_results)

print(f"\nTrain accuracy (no aug):   {train_acc:.4f} ({train_acc*100:.1f}%)")
print(f"Train accuracy (with aug): {train_aug_acc:.4f} ({train_aug_acc*100:.1f}%)")

## Confusion Matrices â€” Train (no aug) vs Train (with aug)

In [ ]:
# Extract arrays from results
train_true = np.array([r['true_label'] for r in train_results])
train_pred = np.array([r['pred_label'] for r in train_results])
train_aug_true = np.array([r['true_label'] for r in train_aug_results])
train_aug_pred = np.array([r['pred_label'] for r in train_aug_results])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, true_arr, pred_arr, title in [
    (axes[0], train_true, train_pred, f'Train no aug (acc={train_acc:.3f})'),
    (axes[1], train_aug_true, train_aug_pred, f'Train with aug (acc={train_aug_acc:.3f})'),
]:
    cm = confusion_matrix(true_arr, pred_arr)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    sns.heatmap(
        cm_norm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=label_encoder.classes,
        yticklabels=label_encoder.classes,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.tight_layout()
display(fig)
plt.close(fig)

# Classification reports
for name, true_arr, pred_arr in [
    ("Train (no aug)", train_true, train_pred),
    ("Train (with aug)", train_aug_true, train_aug_pred),
]:
    print(f"\n--- {name} Classification Report ---")
    print(classification_report(true_arr, pred_arr, target_names=label_encoder.classes, digits=3))

## Build Error Dictionaries

Nested dict: `error_samples[true_label_name][pred_label_name]` â†’ list of result dicts for misclassified tiles.

In [ ]:
from collections import defaultdict

def build_error_dict(results, label_encoder):
    """Build nested dict of misclassified samples: errors[true_name][pred_name] -> list of result dicts."""
    errors = defaultdict(lambda: defaultdict(list))
    for r in results:
        if r['true_label'] != r['pred_label']:
            true_name = label_encoder.decode(r['true_label'])
            pred_name = label_encoder.decode(r['pred_label'])
            errors[true_name][pred_name].append(r)
    # Convert to regular dict for cleaner display
    return {k: dict(v) for k, v in errors.items()}

error_samples_no_aug = build_error_dict(train_results, label_encoder)
error_samples_aug = build_error_dict(train_aug_results, label_encoder)

# Print summary
for name, errors in [("Train (no aug)", error_samples_no_aug), ("Train (with aug)", error_samples_aug)]:
    total_errors = sum(len(v) for sub in errors.values() for v in sub.values())
    print(f"\n{'='*60}")
    print(f"{name}: {total_errors} misclassified tiles")
    print(f"{'='*60}")
    for true_label in sorted(errors.keys()):
        for pred_label in sorted(errors[true_label].keys()):
            count = len(errors[true_label][pred_label])
            print(f"  {true_label:30s} -> {pred_label:30s}: {count} tiles")

## Visualize Misclassified Samples

Pick a true/predicted label pair and visualize N random samples from the error list.

In [ ]:
def visualize_errors(error_dict, dataset, label_encoder, true_label, pred_label,
                     n=5, seed=None):
    """Visualize misclassified tiles as a grid: rows=samples, columns=channels.
    
    Args:
        error_dict: Nested dict from build_error_dict().
        dataset: The TiledMultiChannelDataset used for inference.
        label_encoder: LabelEncoder for decoding labels.
        true_label: True class name string, e.g. 'LPS/Nigericin'.
        pred_label: Predicted class name string, e.g. 'LPS/Unactivated'.
        n: Number of samples to show.
        seed: Random seed for reproducible selection.
    """
    if true_label not in error_dict or pred_label not in error_dict[true_label]:
        print(f"No errors found for '{true_label}' -> '{pred_label}'")
        return
    
    error_list = error_dict[true_label][pred_label]
    total = len(error_list)
    
    # Sample n items
    rng = np.random.RandomState(seed)
    indices = rng.choice(total, size=min(n, total), replace=False)
    selected = [error_list[i] for i in sorted(indices)]
    
    n_show = len(selected)
    n_channels = len(dataset.channels)
    
    fig, axes = plt.subplots(n_show, n_channels, figsize=(3 * n_channels, 3 * n_show))
    if n_show == 1:
        axes = axes[np.newaxis, :]
    
    fig.suptitle(
        f"True: {true_label}  â†’  Predicted: {pred_label}\n"
        f"(showing {n_show}/{total} errors)",
        fontsize=14, fontweight='bold', y=1.02,
    )
    
    # Column headers (channel names)
    for col in range(n_channels):
        axes[0, col].set_title(f"Ch {dataset.channels[col]}", fontsize=11)
    
    for row, r in enumerate(selected):
        tile_idx = r['idx']
        tile_tensor, _ = dataset[tile_idx]
        tile_np = tile_tensor.cpu().numpy()  # (C, H, W)
        
        for col in range(n_channels):
            ax = axes[row, col]
            ax.imshow(tile_np[col], cmap='gray', vmin=0, vmax=1)
            ax.axis('off')
        
        # Row label with metadata
        loc_str = f"{r['well']}/F{r['field']} [{r['top']},{r['left']}]"
        conf_str = f"pred_conf={r['confidence']:.2f}  true_conf={r['true_conf']:.2f}"
        axes[row, 0].set_ylabel(
            f"{loc_str}\n{conf_str}",
            fontsize=9, rotation=0, labelpad=150, va='center',
        )
    
    plt.tight_layout()
    display(fig)
    plt.close(fig)

In [ ]:
# === Example: Visualize errors from Train (no aug) ===
# Pick a true/pred pair from the summary above and set n to how many you want to see

visualize_errors(
    error_dict=error_samples_no_aug,
    dataset=train_dataset,
    label_encoder=label_encoder,
    true_label='LPS/Nigericin',       # <-- change these
    pred_label='Unprimed/Nigericin',     # <-- change these
    n=5,
    seed=22,
)

In [ ]:
# === Example: Visualize errors from Train (with aug) ===
# Compare same error pair but with augmented images to see augmentation effect

visualize_errors(
    error_dict=error_samples_aug,
    dataset=train_dataset_aug,
    label_encoder=label_encoder,
    true_label='LPS/Nigericin',       # <-- change these
    pred_label='Unprimed/Nigericin',     # <-- change these
    n=5,
    seed=42,
)

## ViT Attention Rollout

**Attention Rollout** (Abnar & Zuidema, 2020) traces how attention flows from the final
CLS token back through all transformer layers to the input patches:

1. Extract the attention matrix from **every** block (shape: `NÃ—N`, including CLS + patches)
2. At each layer, add the identity matrix (residual connection) and re-normalize rows
3. Multiply the augmented attention matrices across all layers: `R = A_L * A_{L-1} * ... * A_1`
4. Take the CLS row of the rolled-back matrix â†’ gives per-patch attention coefficients
5. Reshape to 14Ã—14 spatial grid and upsample to image size

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def get_attention_rollout(model, image_tensor, device='cuda', head_fusion='mean',
                          discard_ratio=0.0):
    """Compute attention rollout from all ViT layers for a single image.
    
    Implements the Attention Rollout method (Abnar & Zuidema, 2020):
    - Extract attention weights from ALL transformer blocks
    - Add identity (residual connection) and re-normalize each layer's attention
    - Multiply across layers to get rolled-back attention from CLS to input patches
    
    Args:
        model: TransferLearningClassifier with a ViT backbone.
        image_tensor: Single image tensor of shape (C, H, W).
        device: Device to run inference on.
        head_fusion: How to fuse multi-head attention ('mean', 'max', 'min').
        discard_ratio: Fraction of lowest-attention tokens to zero out per layer
                       before rollout (0.0 = keep all, helps reduce noise).
        
    Returns:
        rollout_map: numpy array of shape (H, W) â€” rolled-back attention heatmap.
        pred_label: int â€” predicted class index.
        probs: numpy array â€” class probabilities.
    """
    backbone = model.model.backbone
    
    # Temporarily disable fused_attn on all blocks
    original_fused = []
    for blk in backbone.blocks:
        original_fused.append(blk.attn.fused_attn)
        blk.attn.fused_attn = False
    
    # Storage for attention weights from each block
    all_attentions = []
    
    def make_hook(storage_list):
        """Create a pre-hook that computes and stores attention weights."""
        def hook_fn(module, input):
            x = input[0]
            B, N, C = x.shape
            qkv = module.qkv(x).reshape(B, N, 3, module.num_heads, module.head_dim).permute(2, 0, 3, 1, 4)
            q, k, v = qkv.unbind(0)
            q, k = module.q_norm(q), module.k_norm(k)
            q = q * module.scale
            attn = q @ k.transpose(-2, -1)
            attn = attn.softmax(dim=-1)
            storage_list.append(attn.cpu().numpy()[0])  # (num_heads, N, N)
        return hook_fn
    
    # Register hooks on ALL blocks
    hooks = []
    for blk in backbone.blocks:
        h = blk.attn.register_forward_pre_hook(make_hook(all_attentions))
        hooks.append(h)
    
    # Forward pass
    img = image_tensor.unsqueeze(0).to(device)
    logits = model(img)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_label = int(logits.argmax(dim=1).cpu().item())
    
    # Remove hooks and restore fused_attn
    for h in hooks:
        h.remove()
    for blk, orig in zip(backbone.blocks, original_fused):
        blk.attn.fused_attn = orig
    
    # --- Attention Rollout ---
    # all_attentions: list of (num_heads, N, N) arrays, one per block
    result = None
    for attn in all_attentions:
        # Fuse heads
        if head_fusion == 'mean':
            attn_fused = attn.mean(axis=0)  # (N, N)
        elif head_fusion == 'max':
            attn_fused = attn.max(axis=0)   # (N, N)
        elif head_fusion == 'min':
            attn_fused = attn.min(axis=0)   # (N, N)
        else:
            raise ValueError(f"Unknown head_fusion: {head_fusion}")
        
        # Optional: discard low-attention tokens (set to 0, re-normalize)
        if discard_ratio > 0:
            flat = attn_fused.flatten()
            threshold = np.quantile(flat, discard_ratio)
            attn_fused[attn_fused < threshold] = 0
            # Re-normalize rows to sum to 1
            row_sums = attn_fused.sum(axis=-1, keepdims=True)
            row_sums[row_sums == 0] = 1  # avoid division by zero
            attn_fused = attn_fused / row_sums
        
        # Add identity matrix (accounts for residual connections)
        N = attn_fused.shape[0]
        attn_fused = 0.5 * attn_fused + 0.5 * np.eye(N)
        # Re-normalize rows
        attn_fused = attn_fused / attn_fused.sum(axis=-1, keepdims=True)
        
        # Multiply with running product
        if result is None:
            result = attn_fused
        else:
            result = attn_fused @ result
    
    # Extract CLS token (row 0) attention to patch tokens (columns 1:)
    cls_attn = result[0, 1:]  # (num_patches,)
    
    # Reshape to spatial grid
    patch_size = 16
    h, w = image_tensor.shape[1], image_tensor.shape[2]
    grid_h, grid_w = h // patch_size, w // patch_size
    rollout_map = cls_attn.reshape(grid_h, grid_w)
    
    # Upsample to original image size
    rollout_tensor = torch.from_numpy(rollout_map).unsqueeze(0).unsqueeze(0).float()
    rollout_up = F.interpolate(rollout_tensor, size=(h, w), mode='bilinear', align_corners=False)
    rollout_np = rollout_up.squeeze().numpy()
    
    # Normalize to [0, 1]
    rollout_np = (rollout_np - rollout_np.min()) / (rollout_np.max() - rollout_np.min() + 1e-8)
    
    return rollout_np, pred_label, probs


@torch.no_grad()
def get_attention_map(model, image_tensor, device='cuda'):
    """Extract attention map from the LAST ViT block only (single-layer, no rollout).
    
    Kept for comparison with rollout method.
    """
    backbone = model.model.backbone
    
    original_fused = []
    for blk in backbone.blocks:
        original_fused.append(blk.attn.fused_attn)
        blk.attn.fused_attn = False
    
    attn_storage = {}
    
    def hook_fn(module, input):
        x = input[0]
        B, N, C = x.shape
        qkv = module.qkv(x).reshape(B, N, 3, module.num_heads, module.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q, k = module.q_norm(q), module.k_norm(k)
        q = q * module.scale
        attn = q @ k.transpose(-2, -1)
        attn = attn.softmax(dim=-1)
        attn_storage['attn'] = attn.cpu().numpy()[0]
    
    hook = backbone.blocks[-1].attn.register_forward_pre_hook(hook_fn)
    
    img = image_tensor.unsqueeze(0).to(device)
    logits = model(img)
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_label = int(logits.argmax(dim=1).cpu().item())
    
    hook.remove()
    for blk, orig in zip(backbone.blocks, original_fused):
        blk.attn.fused_attn = orig
    
    attn = attn_storage['attn']  # (num_heads, N, N)
    cls_attn = attn[:, 0, 1:].mean(axis=0)  # (num_patches,)
    
    patch_size = 16
    h, w = image_tensor.shape[1], image_tensor.shape[2]
    grid_h, grid_w = h // patch_size, w // patch_size
    attn_map = cls_attn.reshape(grid_h, grid_w)
    
    attn_map_tensor = torch.from_numpy(attn_map).unsqueeze(0).unsqueeze(0).float()
    attn_map_up = F.interpolate(attn_map_tensor, size=(h, w), mode='bilinear', align_corners=False)
    attn_map_np = attn_map_up.squeeze().numpy()
    attn_map_np = (attn_map_np - attn_map_np.min()) / (attn_map_np.max() - attn_map_np.min() + 1e-8)
    
    return attn_map_np, pred_label, probs


def visualize_attention(dataset, tile_indices, model, label_encoder, device='cuda',
                        method='rollout', head_fusion='mean', discard_ratio=0.0):
    """Visualize attention maps for given tile indices.
    
    Args:
        dataset: TiledMultiChannelDataset (no augmentation recommended).
        tile_indices: List of tile indices to visualize.
        model: The loaded TransferLearningClassifier.
        label_encoder: LabelEncoder for decoding labels.
        device: Device for inference.
        method: 'rollout' (all layers multiplied) or 'last_layer' (last block only).
        head_fusion: For rollout: 'mean', 'max', or 'min' across heads.
        discard_ratio: For rollout: fraction of low-attention to zero out (0.0-0.9).
    """
    n_samples = len(tile_indices)
    n_channels = len(dataset.channels)
    n_cols = n_channels + 1  # extra column for attention-only map
    
    fig, axes = plt.subplots(n_samples, n_cols, figsize=(3 * n_cols, 3 * n_samples))
    if n_samples == 1:
        axes = axes[np.newaxis, :]
    
    method_label = "Attention Rollout (all layers)" if method == 'rollout' else "Last Layer Attention"
    fig.suptitle(f"ViT {method_label}\nhead_fusion={head_fusion}, discard_ratio={discard_ratio}",
                 fontsize=13, fontweight='bold', y=1.02)
    
    # Column headers
    for col in range(n_channels):
        axes[0, col].set_title(f"Ch {dataset.channels[col]}", fontsize=10)
    axes[0, -1].set_title("Attention", fontsize=10)
    
    for row, tile_idx in enumerate(tile_indices):
        tile_tensor, label = dataset[tile_idx]
        tile_np = tile_tensor.cpu().numpy()  # (C, H, W)
        
        # Get attention map
        if method == 'rollout':
            attn_map, pred_label, probs = get_attention_rollout(
                model, tile_tensor, device, head_fusion=head_fusion,
                discard_ratio=discard_ratio
            )
        else:
            attn_map, pred_label, probs = get_attention_map(model, tile_tensor, device)
        
        true_name = label_encoder.decode(label)
        pred_name = label_encoder.decode(pred_label)
        pred_conf = probs[pred_label]
        
        for col in range(n_channels):
            ax = axes[row, col]
            ax.imshow(tile_np[col], cmap='gray', vmin=0, vmax=1)
            ax.imshow(attn_map, cmap='jet', alpha=0.4, vmin=0, vmax=1)
            ax.axis('off')
        
        # Last column: attention map alone
        ax = axes[row, -1]
        ax.imshow(attn_map, cmap='jet', vmin=0, vmax=1)
        ax.axis('off')
        
        # Row label
        sample_idx, top, left = dataset.tiles[tile_idx]
        sample = dataset.samples[sample_idx]
        loc_str = f"{sample['well']}/F{sample['field']} [{top},{left}]"
        status = "correct" if label == pred_label else "WRONG"
        axes[row, 0].set_ylabel(
            f"[{status}] True: {true_name}\nPred: {pred_name} ({pred_conf:.2f})\n{loc_str}",
            fontsize=8, rotation=0, labelpad=170, va='center',
        )
    
    plt.tight_layout()
    display(fig)
    plt.close(fig)

print("Attention rollout + last-layer utilities ready.")

In [ ]:
# === Attention Rollout on misclassified tiles ===
# Pick a true/pred error pair and visualize rolled-back attention

example_true = 'Unprimed/Nigericin'
example_pred = 'LPS/Nigericin'
n_show = 2

if example_true in error_samples_no_aug and example_pred in error_samples_no_aug[example_true]:
    error_list = error_samples_no_aug[example_true][example_pred]
    rng = np.random.RandomState(42)
    selected = rng.choice(len(error_list), size=n_show, replace=False)
    tile_indices = [error_list[i]['idx'] for i in sorted(selected)]
    
    print(f"Attention ROLLOUT for {n_show} misclassified tiles:")
    print(f"  True: {example_true} â†’ Predicted: {example_pred}\n")
    
    # Rollout (all layers multiplied back to input)
    visualize_attention(train_dataset, tile_indices, model, label_encoder, device,
                        method='rollout', head_fusion='mean', discard_ratio=0.0)
else:
    print(f"No errors found for '{example_true}' -> '{example_pred}'")

## Pretrained (Untrained) ViT-S — Attention Comparison

Load a ViT-S with ImageNet pretrained weights from timm (no fine-tuning on this dataset).
Compare where the pretrained model attends vs. where the fine-tuned model attends on the
same tiles, to see what fine-tuning actually changed in the attention patterns.

In [ ]:
import timm

# Load pretrained ViT-S from timm with same in_channels but NO fine-tuning
pretrained_vit = timm.create_model(
    'vit_small_patch16_224',
    pretrained=True,
    num_classes=0,       # feature extractor only (no classifier head)
    in_chans=len(CHANNELS),
)
pretrained_vit = pretrained_vit.to(device)
pretrained_vit.eval()

print(f"Pretrained ViT-S loaded: {sum(p.numel() for p in pretrained_vit.parameters()):,} params")
print(f"  in_chans={len(CHANNELS)}, patch_size=16, image_size=224")
print(f"  num_blocks={len(pretrained_vit.blocks)}, embed_dim={pretrained_vit.embed_dim}")
print("  NOTE: ImageNet weights only — NOT fine-tuned on this dataset")

In [ ]:
@torch.no_grad()
def get_attention_rollout_raw(backbone, image_tensor, device='cuda',
                              head_fusion='mean', discard_ratio=0.0):
    """Attention rollout for a raw timm ViT backbone (no classifier wrapper).
    
    Same algorithm as get_attention_rollout but works directly on a timm ViT
    model (e.g. timm.create_model('vit_small_patch16_224', num_classes=0)).
    
    Returns:
        rollout_map: numpy array (H, W) — attention heatmap normalized to [0,1].
    """
    original_fused = []
    for blk in backbone.blocks:
        original_fused.append(blk.attn.fused_attn)
        blk.attn.fused_attn = False
    
    all_attentions = []
    
    def make_hook(storage_list):
        def hook_fn(module, input):
            x = input[0]
            B, N, C = x.shape
            qkv = module.qkv(x).reshape(B, N, 3, module.num_heads, module.head_dim).permute(2, 0, 3, 1, 4)
            q, k, v = qkv.unbind(0)
            q, k = module.q_norm(q), module.k_norm(k)
            q = q * module.scale
            attn = q @ k.transpose(-2, -1)
            attn = attn.softmax(dim=-1)
            storage_list.append(attn.cpu().numpy()[0])
        return hook_fn
    
    hooks = []
    for blk in backbone.blocks:
        h = blk.attn.register_forward_pre_hook(make_hook(all_attentions))
        hooks.append(h)
    
    img = image_tensor.unsqueeze(0).to(device)
    _ = backbone(img)  # forward pass (output is features, not logits)
    
    for h in hooks:
        h.remove()
    for blk, orig in zip(backbone.blocks, original_fused):
        blk.attn.fused_attn = orig
    
    # Rollout: multiply attention matrices across layers
    result = None
    for attn in all_attentions:
        if head_fusion == 'mean':
            attn_fused = attn.mean(axis=0)
        elif head_fusion == 'max':
            attn_fused = attn.max(axis=0)
        elif head_fusion == 'min':
            attn_fused = attn.min(axis=0)
        else:
            raise ValueError(f"Unknown head_fusion: {head_fusion}")
        
        if discard_ratio > 0:
            flat = attn_fused.flatten()
            threshold = np.quantile(flat, discard_ratio)
            attn_fused[attn_fused < threshold] = 0
            row_sums = attn_fused.sum(axis=-1, keepdims=True)
            row_sums[row_sums == 0] = 1
            attn_fused = attn_fused / row_sums
        
        N = attn_fused.shape[0]
        attn_fused = 0.5 * attn_fused + 0.5 * np.eye(N)
        attn_fused = attn_fused / attn_fused.sum(axis=-1, keepdims=True)
        
        if result is None:
            result = attn_fused
        else:
            result = attn_fused @ result
    
    cls_attn = result[0, 1:]
    patch_size = 16
    h, w = image_tensor.shape[1], image_tensor.shape[2]
    grid_h, grid_w = h // patch_size, w // patch_size
    rollout_map = cls_attn.reshape(grid_h, grid_w)
    
    rollout_tensor = torch.from_numpy(rollout_map).unsqueeze(0).unsqueeze(0).float()
    rollout_up = F.interpolate(rollout_tensor, size=(h, w), mode='bilinear', align_corners=False)
    rollout_np = rollout_up.squeeze().numpy()
    rollout_np = (rollout_np - rollout_np.min()) / (rollout_np.max() - rollout_np.min() + 1e-8)
    
    return rollout_np

print("Raw timm ViT attention rollout function ready.")

In [ ]:
def visualize_attention_comparison(dataset, tile_indices, finetuned_model, pretrained_backbone,
                                   label_encoder, device='cuda',
                                   head_fusion='mean', discard_ratio=0.0):
    """Side-by-side attention rollout: fine-tuned vs pretrained (untrained) ViT.
    
    For each tile: one row with columns = [channels w/ fine-tuned attn] + [fine-tuned only] + [pretrained only].
    
    Args:
        dataset: TiledMultiChannelDataset (no augmentation recommended).
        tile_indices: List of tile indices to visualize.
        finetuned_model: The fine-tuned TransferLearningClassifier.
        pretrained_backbone: Raw timm ViT backbone (num_classes=0).
        label_encoder: LabelEncoder for decoding labels.
        device: Device for inference.
        head_fusion: 'mean', 'max', or 'min' across heads.
        discard_ratio: Fraction of low-attention to zero out (0.0-0.9).
    """
    n_samples = len(tile_indices)
    n_channels = len(dataset.channels)
    # Columns: [ch1..chN with fine-tuned overlay] + [fine-tuned attn] + [pretrained attn]
    n_cols = n_channels + 2
    
    fig, axes = plt.subplots(n_samples, n_cols, figsize=(3 * n_cols, 3 * n_samples))
    if n_samples == 1:
        axes = axes[np.newaxis, :]
    
    fig.suptitle(
        f"Attention Rollout Comparison: Fine-tuned vs Pretrained (ImageNet)\n"
        f"head_fusion={head_fusion}, discard_ratio={discard_ratio}",
        fontsize=13, fontweight='bold', y=1.02,
    )
    
    for col in range(n_channels):
        axes[0, col].set_title(f"Ch {dataset.channels[col]}\n(fine-tuned)", fontsize=9)
    axes[0, -2].set_title("Fine-tuned\nAttention", fontsize=9)
    axes[0, -1].set_title("Pretrained\nAttention", fontsize=9)
    
    for row, tile_idx in enumerate(tile_indices):
        tile_tensor, label = dataset[tile_idx]
        tile_np = tile_tensor.cpu().numpy()
        
        # Fine-tuned attention
        ft_attn, pred_label, probs = get_attention_rollout(
            finetuned_model, tile_tensor, device,
            head_fusion=head_fusion, discard_ratio=discard_ratio,
        )
        # Pretrained attention
        pt_attn = get_attention_rollout_raw(
            pretrained_backbone, tile_tensor, device,
            head_fusion=head_fusion, discard_ratio=discard_ratio,
        )
        
        true_name = label_encoder.decode(label)
        pred_name = label_encoder.decode(pred_label)
        pred_conf = probs[pred_label]
        
        # Channel columns with fine-tuned overlay
        for col in range(n_channels):
            ax = axes[row, col]
            ax.imshow(tile_np[col], cmap='gray', vmin=0, vmax=1)
            ax.imshow(ft_attn, cmap='jet', alpha=0.4, vmin=0, vmax=1)
            ax.axis('off')
        
        # Fine-tuned attention only
        axes[row, -2].imshow(ft_attn, cmap='jet', vmin=0, vmax=1)
        axes[row, -2].axis('off')
        
        # Pretrained attention only
        axes[row, -1].imshow(pt_attn, cmap='jet', vmin=0, vmax=1)
        axes[row, -1].axis('off')
        
        # Row label
        sample_idx, top, left = dataset.tiles[tile_idx]
        sample = dataset.samples[sample_idx]
        loc_str = f"{sample['well']}/F{sample['field']} [{top},{left}]"
        status = "correct" if label == pred_label else "WRONG"
        axes[row, 0].set_ylabel(
            f"[{status}] True: {true_name}\nPred: {pred_name} ({pred_conf:.2f})\n{loc_str}",
            fontsize=8, rotation=0, labelpad=170, va='center',
        )
    
    plt.tight_layout()
    display(fig)
    plt.close(fig)

print("Comparison visualization ready.")

In [ ]:
# === Compare fine-tuned vs pretrained attention on misclassified tiles ===

example_true = 'Unprimed/Nigericin'
example_pred = 'LPS/Nigericin'
n_show = 2

if example_true in error_samples_no_aug and example_pred in error_samples_no_aug[example_true]:
    error_list = error_samples_no_aug[example_true][example_pred]
    rng = np.random.RandomState(22)
    selected = rng.choice(len(error_list), size=n_show, replace=False)
    tile_indices = [error_list[i]['idx'] for i in sorted(selected)]
    
    print(f"Fine-tuned vs Pretrained attention for {n_show} misclassified tiles:")
    print(f"  True: {example_true} → Predicted: {example_pred}\n")
    
    visualize_attention_comparison(
        train_dataset, tile_indices, model, pretrained_vit, label_encoder, device,
        head_fusion='mean', discard_ratio=0.0,
    )
else:
    print(f"No errors found for '{example_true}' -> '{example_pred}'")